<!-- paired-role-banner -->
> 🟩 **정답 코드 · 오른쪽 창** — 왼쪽과 같은 셀 위치의 참고 구현입니다. 필요한 한 줄만 확인하고 다시 실습 창으로 돌아가세요.

# 20. 표형 MLP와 비지도 센서 이상 탐지

희소한 계정 위험 레이블을 MLP로 분류하고 threshold를 validation에서 선택합니다.
이어서 센서 정상 구간만으로 autoencoder를 학습해 reconstruction error 기반 이상 탐지를
구성합니다. 지도/비지도 접근의 데이터 요구 사항과 평가 차이를 비교합니다.

In [ ]:
# 이 컴퓨터를 감지해 안전한 시작값을 확인합니다. 각 실습은 더 작은 값으로도 동작합니다.
from llm_engineering_lab.hardware import HardwareProfile, recommend_training_profile

LAB_HARDWARE = HardwareProfile.detect()
LAB_PROFILE = recommend_training_profile(LAB_HARDWARE)
print("hardware:", LAB_HARDWARE)
print("recommended profile:", LAB_PROFILE)

## 학습 목표

- train 통계만 사용하는 숫자 표준화와 범주 one-hot 파이프라인을 만든다.
- `pos_weight`로 class imbalance를 손실에 반영한다.
- 0.5 고정 대신 validation F1로 threshold를 고른다.
- 정상 데이터만으로 autoencoder를 fit하고 validation error로 임계값을 정한다.
- accuracy가 희소 이벤트에서 왜 위험한 지표인지 설명한다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=random.seed,np.random.seed,torch.manual_seed,torch.cuda.manual_seed_all -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 4개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>random.seed(a=None)</code></summary>

#### `random.seed(a=None)`

- **역할:** Python 표준 난수 생성기의 상태를 초기화합니다.
- **입력·반환:** 정수 등의 시드를 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 같은 Python 난수 수열을 다시 만들어 실험을 재현합니다.
- **주의점:** NumPy와 PyTorch의 난수 상태는 별도로 고정해야 합니다.
- **공식 문서:** [random.seed](https://docs.python.org/3/library/random.html#random.seed)

</details>

<details>
<summary><code>np.random.seed(seed=None)</code></summary>

#### `np.random.seed(seed=None)`

- **역할:** NumPy의 기존 전역 난수 생성기 상태를 초기화합니다.
- **입력·반환:** 시드를 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 레거시 `np.random.*` 호출의 수열을 재현합니다.
- **주의점:** 새 코드에서는 독립 상태를 가진 `default_rng`가 권장됩니다.
- **공식 문서:** [np.random.seed](https://numpy.org/doc/stable/reference/random/generated/numpy.random.seed.html)

</details>

<details>
<summary><code>torch.manual_seed(seed)</code></summary>

#### `torch.manual_seed(seed)`

- **역할:** PyTorch 난수 생성기의 시드를 설정합니다.
- **입력·반환:** 정수 시드를 받고 `Generator`를 반환합니다.
- **이 셀에서 쓰는 이유:** 가중치 초기화와 텐서 샘플링을 반복 가능하게 만듭니다.
- **주의점:** 완전한 결정성에는 알고리즘·장치 설정도 추가로 필요할 수 있습니다.
- **공식 문서:** [torch.manual_seed](https://docs.pytorch.org/docs/stable/generated/torch.manual_seed.html)

</details>

<details>
<summary><code>torch.cuda.manual_seed_all(seed)</code></summary>

#### `torch.cuda.manual_seed_all(seed)`

- **역할:** 사용 가능한 모든 CUDA GPU의 난수 시드를 설정합니다.
- **입력·반환:** 정수 시드를 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 여러 GPU에서도 초기화와 샘플링을 최대한 반복 가능하게 합니다.
- **주의점:** CPU 시드와 결정론 설정을 대신하지 않습니다.
- **공식 문서:** [torch.cuda.manual_seed_all](https://docs.pytorch.org/docs/stable/generated/torch.cuda.manual_seed_all.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
import random
import numpy as np
import pandas as pd
import torch
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score, precision_recall_fscore_support, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from llm_engineering_lab.acceleration import get_accelerator

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
ACCELERATOR = get_accelerator()
device = ACCELERATOR.device
print(ACCELERATOR.summary())

In [ ]:
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data" / "practice").exists():
            return candidate
    raise FileNotFoundError(
        "프로젝트 루트를 찾지 못했습니다. Start_JupyterLab.cmd로 실행해 주세요."
    )


ROOT = find_project_root()
DATA = ROOT / "data" / "practice"
ARTIFACTS = ROOT / "artifacts" / "practice"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
print("project root:", ROOT)

## 1) 희소한 계정 risk 데이터

accuracy만 보면 모든 샘플을 정상으로 예측하는 모델도 높아 보일 수 있습니다.
positive 비율과 naive baseline을 먼저 계산한 뒤 precision/recall/F1을 사용합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=pd.read_csv -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>pd.read_csv(filepath_or_buffer, ...)</code></summary>

#### `pd.read_csv(filepath_or_buffer, ...)`

- **역할:** CSV 표 데이터를 DataFrame으로 읽습니다.
- **입력·반환:** 표 데이터와 열·그룹 설정을 받고 DataFrame, Series 또는 집계 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 학습 전 데이터 분포·결측·중복과 분할 기준을 점검합니다.
- **주의점:** dtype 추론, 결측값, index 보존과 grouping 후 shape 변화를 확인해야 합니다.
- **공식 문서:** [pd.read_csv](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
frame = pd.read_csv(DATA / "tabular_risk.csv")
target = "risk_label"
identifier = "account_id"
print(frame.shape, frame.dtypes.to_dict())
positive_rate = frame[target].mean()
print(
    "positive rate:",
    round(positive_rate, 4),
    "all-negative accuracy:",
    round(1 - positive_rate, 4),
)

<!-- api-explanations:start -->
<!-- api-explanations:keys=train_test_split -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>train_test_split(*arrays, test_size=None, random_state=None, ...)</code></summary>

#### `train_test_split(*arrays, test_size=None, random_state=None, ...)`

- **역할:** 배열들을 같은 index 기준으로 train과 test로 나눕니다.
- **입력·반환:** 하나 이상의 배열을 받고 각 배열의 train·test 조각을 순서대로 반환합니다.
- **이 셀에서 쓰는 이유:** 평가 데이터 누수를 막고 동일 분할을 재현합니다.
- **주의점:** 분류에서는 `stratify`와 `random_state` 필요 여부를 확인해야 합니다.
- **공식 문서:** [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
train_frame, temp_frame = train_test_split(
    frame, test_size=0.30, random_state=SEED, stratify=frame[target]
)
valid_frame, test_frame = train_test_split(
    temp_frame, test_size=0.50, random_state=SEED, stratify=temp_frame[target]
)
numeric = [
    "tenure_months",
    "monthly_spend",
    "weekly_sessions",
    "support_tickets_90d",
    "latency_ms",
]
categorical = ["industry", "region", "tier"]
preprocessor = ColumnTransformer(
    [
        (
            "numeric",
            Pipeline(
                [
                    ("impute", SimpleImputer(strategy="median")),
                    ("scale", StandardScaler()),
                ]
            ),
            numeric,
        ),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical,
        ),
    ]
)
X_train = preprocessor.fit_transform(train_frame).astype(np.float32)
X_valid = preprocessor.transform(valid_frame).astype(np.float32)
X_test = preprocessor.transform(test_frame).astype(np.float32)
y_train = train_frame[target].to_numpy(np.float32)
y_valid = valid_frame[target].to_numpy(np.int64)
y_test = test_frame[target].to_numpy(np.int64)
print(X_train.shape, X_valid.shape, X_test.shape)

<!-- api-explanations:start -->
<!-- api-explanations:keys=np.isfinite,Tensor.mean -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>np.isfinite(x)</code></summary>

#### `np.isfinite(x)`

- **역할:** 각 값이 NaN이나 무한대가 아닌 유한수인지 검사합니다.
- **입력·반환:** 배열형 입력을 받고 같은 shape의 불리언 배열을 반환합니다.
- **이 셀에서 쓰는 이유:** 학습 값이나 전처리 결과가 수치적으로 정상인지 검증합니다.
- **주의점:** 유한하다는 사실만 확인하며 값의 크기가 적절하다는 뜻은 아닙니다.
- **공식 문서:** [np.isfinite](https://numpy.org/doc/stable/reference/generated/numpy.isfinite.html)

</details>

<details>
<summary><code>tensor.mean(dim=None, keepdim=False, dtype=None)</code></summary>

#### `tensor.mean(dim=None, keepdim=False, dtype=None)`

- **역할:** 지정 축의 산술평균을 계산합니다.
- **입력·반환:** 텐서와 축 설정을 받고 축약된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** batch loss, feature 통계, metric을 하나의 대표값으로 축약합니다.
- **주의점:** 어느 축을 없애는지와 빈 텐서에서 NaN이 나올 수 있음을 확인합니다.

##### 관련 수식과 코드 연결

$$
\bar x=\frac{1}{N}\sum_{i=1}^{N}x_i
$$

- **기호:** $N$은 선택한 축의 원소 수입니다.
- **수식 → 코드:** `dim` 또는 `axis`가 평균에 포함할 원소와 결과에서 사라질 축을 결정합니다.
- **직관:** 여러 관측값을 중심 위치 하나로 요약합니다.
- **shape 확인:** `keepdim=False`이면 축약한 축이 제거되고 참이면 크기 1로 남습니다.
- **공식 문서:** [Tensor.mean](https://docs.pytorch.org/docs/stable/generated/torch.mean.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
assert X_train.dtype == X_valid.dtype == X_test.dtype == np.float32
assert X_train.shape[1] == X_valid.shape[1] == X_test.shape[1]
assert (
    np.isfinite(X_train).all()
    and np.isfinite(X_valid).all()
    and np.isfinite(X_test).all()
)
assert abs(y_train.mean() - y_test.mean()) < 0.03
print("preprocessing contract: OK")

## 2) class imbalance를 고려한 MLP

`BCEWithLogitsLoss(pos_weight=negative/positive)`는 양성 오류를 더 크게 벌점화합니다.
확률을 먼저 sigmoid한 뒤 BCE를 적용하지 말고, 수치적으로 안정적인 logits 손실을 사용합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=nn.Sequential,nn.Linear,nn.ReLU,nn.Dropout,Tensor.sum,torch.tensor,torch.optim.AdamW,Module.parameters,Tensor.item -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 9개 · 수식 5개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>nn.Sequential(*modules)</code></summary>

#### `nn.Sequential(*modules)`

- **역할:** 모듈을 순서대로 연결합니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.
- **공식 문서:** [nn.Sequential](https://docs.pytorch.org/docs/stable/generated/torch.nn.Sequential.html)

</details>

<details>
<summary><code>nn.Linear(in_features, out_features, bias=True)</code></summary>

#### `nn.Linear(in_features, out_features, bias=True)`

- **역할:** 완전연결 affine 변환을 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
y=xW^{\mathsf T}+b
$$

- **기호:** $x$는 입력, $W$는 weight, $b$는 bias, $y$는 출력입니다.
- **수식 → 코드:** `in_features`와 `out_features`가 각각 $W$의 두 축을 결정합니다.
- **직관:** 각 출력 feature는 모든 입력 feature의 학습 가능한 가중합입니다.
- **shape 확인:** `[..., D_in] -> [..., D_out]`, weight는 `[D_out, D_in]`입니다.
- **공식 문서:** [nn.Linear](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html)

</details>

<details>
<summary><code>nn.ReLU(inplace=False)</code></summary>

#### `nn.ReLU(inplace=False)`

- **역할:** 음수 값을 0으로 만드는 활성함수를 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\operatorname{ReLU}(x)=\max(0,x)
$$

- **기호:** $x$는 선형 변환 뒤 activation입니다.
- **수식 → 코드:** 코드가 음수 원소를 0으로 치환하고 양수는 그대로 통과시킵니다.
- **직관:** 단순하지만 여러 선형 층을 비선형 함수로 조합할 수 있게 합니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [nn.ReLU](https://docs.pytorch.org/docs/stable/generated/torch.nn.ReLU.html)

</details>

<details>
<summary><code>nn.Dropout(p=0.5)</code></summary>

#### `nn.Dropout(p=0.5)`

- **역할:** 학습 중 일부 activation을 무작위로 0으로 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
y_i=\frac{m_i}{1-p}x_i,\qquad m_i\sim\operatorname{Bernoulli}(1-p)
$$

- **기호:** $p$는 제거 확률, $m_i$는 학습 중 표본화한 mask입니다.
- **수식 → 코드:** `model.train()`일 때 mask와 $1/(1-p)$ scale을 적용하고 `eval()`에서는 그대로 둡니다.
- **직관:** 일부 activation에 의존하지 못하게 하면서 기댓값은 유지합니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [nn.Dropout](https://docs.pytorch.org/docs/stable/generated/torch.nn.Dropout.html)

</details>

<details>
<summary><code>tensor.sum(dim=None, keepdim=False, dtype=None)</code></summary>

#### `tensor.sum(dim=None, keepdim=False, dtype=None)`

- **역할:** 지정 축의 원소 합을 계산합니다.
- **입력·반환:** 텐서와 축 설정을 받고 축약된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 확률 질량, mask 개수, message aggregation을 계산합니다.
- **주의점:** 정수 overflow와 축약 뒤 shape 변화를 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
S=\sum_{i=1}^{N}x_i
$$

- **기호:** $N$은 선택 축의 원소 수입니다.
- **수식 → 코드:** `dim` 또는 `axis`의 모든 원소를 더해 축약합니다.
- **직관:** count, 확률 질량, 이웃 message처럼 누적량을 계산합니다.
- **shape 확인:** 선택한 축은 제거되거나 `keepdim=True`이면 크기 1로 남습니다.
- **공식 문서:** [Tensor.sum](https://docs.pytorch.org/docs/stable/generated/torch.sum.html)

</details>

<details>
<summary><code>torch.tensor(data, dtype=None, device=None)</code></summary>

#### `torch.tensor(data, dtype=None, device=None)`

- **역할:** 데이터를 복사해 새 PyTorch 텐서를 만듭니다.
- **입력·반환:** 배열형 데이터와 선택적 dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 상수·label·축소 실험 입력을 명시적으로 구성합니다.
- **주의점:** 기존 텐서에서 호출하면 복사가 생기므로 경우에 따라 `as_tensor`가 낫습니다.
- **공식 문서:** [torch.tensor](https://docs.pytorch.org/docs/stable/generated/torch.tensor.html)

</details>

<details>
<summary><code>torch.optim.AdamW(params, lr=1e-3, weight_decay=1e-2, ...)</code></summary>

#### `torch.optim.AdamW(params, lr=1e-3, weight_decay=1e-2, ...)`

- **역할:** weight decay를 gradient update와 분리한 AdamW optimizer를 만듭니다.
- **입력·반환:** 파라미터와 hyperparameter를 받고 optimizer 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** Transformer 계열에서 흔한 최적화 규칙을 재현합니다.
- **주의점:** bias·정규화 파라미터를 decay에서 제외하는 설정도 자주 사용합니다.

##### 관련 수식과 코드 연결

$$
\theta_t=(1-\eta\lambda)\theta_{t-1}-\eta\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}
$$

- **기호:** $\lambda$는 weight decay이고 나머지 기호는 Adam과 같습니다.
- **수식 → 코드:** 코드의 `weight_decay`가 gradient 기반 Adam update와 분리된 축소 항으로 적용됩니다.
- **직관:** 손실 gradient와 무관하게 큰 weight를 조금씩 줄이는 것이 Adam과의 핵심 차이입니다.
- **shape 확인:** 파라미터와 모멘트 tensor의 shape은 변하지 않습니다.
- **공식 문서:** [torch.optim.AdamW](https://docs.pytorch.org/docs/stable/generated/torch.optim.AdamW.html)

</details>

<details>
<summary><code>model.parameters(recurse=True)</code></summary>

#### `model.parameters(recurse=True)`

- **역할:** 모듈에 등록된 학습 파라미터 iterator를 반환합니다.
- **입력·반환:** 하위 모듈 포함 여부를 받고 `Parameter` iterator를 반환합니다.
- **이 셀에서 쓰는 이유:** optimizer가 갱신할 weight와 bias를 전달합니다.
- **주의점:** iterator는 한 번 소비되며 buffer와 일반 tensor attribute는 포함하지 않습니다.
- **공식 문서:** [Module.parameters](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>tensor.item()</code></summary>

#### `tensor.item()`

- **역할:** 원소 하나짜리 텐서에서 Python 숫자를 꺼냅니다.
- **입력·반환:** 단일 원소 텐서를 받고 Python 숫자를 반환합니다.
- **이 셀에서 쓰는 이유:** loss나 metric을 출력·기록 가능한 값으로 바꿉니다.
- **주의점:** 여러 원소 텐서에는 쓸 수 없고 GPU에서는 동기화를 일으킬 수 있습니다.
- **공식 문서:** [Tensor.item](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.item.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
class RiskMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=32):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.10),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, x):
        return self.network(x).squeeze(1)


model = RiskMLP(X_train.shape[1]).to(device)
positives = float(y_train.sum())
pos_weight = torch.tensor([(len(y_train) - positives) / positives], device=device)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-4)
print("pos_weight:", pos_weight.item())

<!-- api-explanations:start -->
<!-- api-explanations:keys=DataLoader,torch.from_numpy -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>DataLoader(dataset, batch_size=1, shuffle=False, ...)</code></summary>

#### `DataLoader(dataset, batch_size=1, shuffle=False, ...)`

- **역할:** dataset에서 mini-batch를 반복 생성합니다.
- **입력·반환:** dataset과 batching 설정을 받고 iterable loader를 반환합니다.
- **이 셀에서 쓰는 이유:** 학습 데이터를 batch 단위로 섞고 모델에 공급합니다.
- **주의점:** Windows 다중 worker는 진입점 보호가 필요하며 마지막 batch 크기도 확인합니다.
- **공식 문서:** [DataLoader](https://docs.pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader)

</details>

<details>
<summary><code>torch.from_numpy(ndarray)</code></summary>

#### `torch.from_numpy(ndarray)`

- **역할:** NumPy 배열과 메모리를 공유하는 CPU 텐서를 만듭니다.
- **입력·반환:** NumPy `ndarray`를 받고 CPU 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 복사 없이 준비된 NumPy 데이터를 모델 입력으로 바꿉니다.
- **주의점:** 한쪽의 in-place 변경이 다른 쪽에도 반영될 수 있습니다.
- **공식 문서:** [torch.from_numpy](https://docs.pytorch.org/docs/stable/generated/torch.from_numpy.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
train_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train)),
    batch_size=64,
    shuffle=True,
)
print(
    "batches:",
    len(train_loader),
    "parameters:",
    sum(p.numel() for p in model.parameters()),
)

<!-- api-explanations:start -->
<!-- api-explanations:keys=Module.train,Tensor.to,Optimizer.zero_grad,Tensor.backward,Optimizer.step,torch.inference_mode,Module.eval -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 7개 · 수식 2개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>model.train(mode=True)</code></summary>

#### `model.train(mode=True)`

- **역할:** 모듈과 하위 모듈을 학습 모드로 전환합니다.
- **입력·반환:** 불리언 모드를 받고 모듈 자신을 반환합니다.
- **이 셀에서 쓰는 이유:** Dropout과 BatchNorm이 학습 시 동작을 사용하도록 합니다.
- **주의점:** gradient를 켜는 함수가 아니므로 `no_grad`와 독립적입니다.
- **공식 문서:** [Module.train](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>tensor.to(device=None, dtype=None)</code></summary>

#### `tensor.to(device=None, dtype=None)`

- **역할:** 텐서를 지정 장치나 dtype으로 변환합니다.
- **입력·반환:** 장치·dtype을 받고 변환된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 모델과 입력의 장치·정밀도를 일치시킵니다.
- **주의점:** 원본과 조건이 같으면 복사 없이 같은 텐서를 돌려줄 수 있습니다.
- **공식 문서:** [Tensor.to](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.to.html)

</details>

<details>
<summary><code>optimizer.zero_grad(set_to_none=True)</code></summary>

#### `optimizer.zero_grad(set_to_none=True)`

- **역할:** 파라미터에 누적된 gradient를 비웁니다.
- **입력·반환:** 초기화 방식을 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 현재 mini-batch의 gradient만으로 update하도록 학습 단계를 시작합니다.
- **주의점:** 호출하지 않으면 PyTorch gradient가 기본적으로 누적됩니다.
- **공식 문서:** [Optimizer.zero_grad](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.zero_grad.html)

</details>

<details>
<summary><code>loss.backward()</code></summary>

#### `loss.backward()`

- **역할:** 스칼라 loss에서 계산 그래프를 거슬러 gradient를 계산합니다.
- **입력·반환:** 스칼라 텐서에서는 보통 인자 없이 호출하며 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** optimizer update에 필요한 각 파라미터의 `.grad`를 채웁니다.
- **주의점:** 같은 그래프를 다시 역전파하려면 별도 보존 설정이 필요합니다.

##### 관련 수식과 코드 연결

$$
\frac{\partial L}{\partial x}=\frac{\partial L}{\partial y}\frac{\partial y}{\partial x},\qquad x.\mathrm{grad}\leftarrow x.\mathrm{grad}+\nabla_xL
$$

- **기호:** $L$은 scalar loss이고 $x$는 계산 그래프의 leaf parameter입니다.
- **수식 → 코드:** `backward()`는 연쇄법칙으로 gradient를 계산해 각 parameter의 `.grad`에 누적합니다.
- **직관:** 파라미터를 직접 바꾸는 호출이 아니라 update에 필요한 미분값을 준비하는 호출입니다.
- **shape 확인:** 각 `.grad`는 대응 parameter와 같은 shape이며 scalar가 아닌 출력에는 seed gradient가 필요합니다.
- **공식 문서:** [Tensor.backward](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.backward.html)

</details>

<details>
<summary><code>optimizer.step(closure=None)</code></summary>

#### `optimizer.step(closure=None)`

- **역할:** 현재 gradient와 optimizer 규칙으로 파라미터를 갱신합니다.
- **입력·반환:** 선택적 closure를 받고 optimizer에 따라 loss 또는 `None`을 반환합니다.
- **이 셀에서 쓰는 이유:** 역전파로 계산한 gradient를 실제 학습 update에 반영합니다.
- **주의점:** 일반적으로 `zero_grad`, forward, `backward` 다음에 호출합니다.

##### 관련 수식과 코드 연결

$$
\theta_{t+1}=\mathcal{U}(\theta_t,\nabla_\theta L_t,s_t;\eta)
$$

- **기호:** $\mathcal U$는 optimizer update, $s_t$는 momentum·moment 상태입니다.
- **수식 → 코드:** `step()`은 이미 `.grad`에 저장된 값을 읽어 선택한 optimizer 규칙을 적용합니다.
- **직관:** gradient 계산과 parameter 갱신을 분리하면 학습 lifecycle을 정확히 추적할 수 있습니다.
- **shape 확인:** parameter shape은 유지되고 값과 optimizer state만 바뀝니다.
- **공식 문서:** [Optimizer.step](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.step.html)

</details>

<details>
<summary><code>torch.inference_mode(mode=True)</code></summary>

#### `torch.inference_mode(mode=True)`

- **역할:** 추론 전용으로 autograd 관련 추적을 더 강하게 비활성화합니다.
- **입력·반환:** 불리언 모드를 받고 context manager 또는 decorator로 동작합니다.
- **이 셀에서 쓰는 이유:** 순수 평가 구간의 오버헤드를 줄입니다.
- **주의점:** 이 모드에서 만든 텐서는 이후 gradient 계산에 쓰기 어려울 수 있습니다.
- **공식 문서:** [torch.inference_mode](https://docs.pytorch.org/docs/stable/generated/torch.inference_mode.html)

</details>

<details>
<summary><code>model.eval()</code></summary>

#### `model.eval()`

- **역할:** 모듈과 하위 모듈을 평가 모드로 전환합니다.
- **입력·반환:** 인자 없이 호출하고 모듈 자신을 반환합니다.
- **이 셀에서 쓰는 이유:** Dropout을 끄고 BatchNorm의 저장 통계를 사용하게 합니다.
- **주의점:** gradient 기록은 끄지 않으므로 필요하면 `no_grad`도 사용합니다.
- **공식 문서:** [Module.eval](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
for epoch in range(10):
    model.train()
    total = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        loss = loss_fn(model(x), y)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(y)
    if epoch in {0, 4, 9}:
        print(f"epoch={epoch + 1:02d} loss={total / len(train_loader.dataset):.4f}")


@torch.inference_mode()
def predict_probability(model, features):
    model.eval()
    x = torch.from_numpy(features).to(device)
    return model(x).sigmoid().cpu().numpy()


valid_probability = predict_probability(model, X_valid)

## 3) threshold는 validation에서 선택

손실의 class weight와 의사결정 threshold는 별개입니다. 운영 비용에 따라 precision과
recall의 선호가 달라집니다. 여기서는 단순화를 위해 validation F1 최대값을 선택합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=np.array,Tensor.argmax -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>np.array(object, dtype=None, copy=True, ...)</code></summary>

#### `np.array(object, dtype=None, copy=True, ...)`

- **역할:** 입력 데이터로 새 NumPy 배열을 만듭니다.
- **입력·반환:** 배열과 축·dtype 등의 설정을 받고 배열 또는 수치 결과를 반환합니다.
- **이 셀에서 쓰는 이유:** 데이터 shape과 수치 연산을 논문 실험에 맞게 구성·검증합니다.
- **주의점:** 축, dtype, 복사·view 여부와 빈 배열에서의 동작을 확인해야 합니다.
- **공식 문서:** [np.array](https://numpy.org/doc/stable/reference/generated/numpy.array.html)

</details>

<details>
<summary><code>tensor.argmax(dim=None, keepdim=False)</code></summary>

#### `tensor.argmax(dim=None, keepdim=False)`

- **역할:** 지정 축에서 최댓값의 index를 찾습니다.
- **입력·반환:** 텐서와 축을 받고 정수 index 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** logit을 예측 class나 선택 action으로 바꿉니다.
- **주의점:** 미분 불가능하며 동점에서는 첫 index가 선택됩니다.

##### 관련 수식과 코드 연결

$$
k^\star=\operatorname*{arg\,max}_{k}x_k
$$

- **기호:** $k^\star$는 가장 큰 값 자체가 아니라 그 값의 index입니다.
- **수식 → 코드:** `dim` 또는 `axis`가 어느 후보 집합에서 최댓값 index를 고를지 정합니다.
- **직관:** 연속 score를 이산 class·action 결정으로 바꾸는 선택 연산입니다.
- **shape 확인:** 선택 축이 제거되며 미분 가능한 gradient 경로는 만들지 않습니다.
- **공식 문서:** [Tensor.argmax](https://docs.pytorch.org/docs/stable/generated/torch.argmax.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
thresholds = np.linspace(0.10, 0.90, 81)
valid_f1 = np.array(
    [
        f1_score(y_valid, valid_probability >= threshold, zero_division=0)
        for threshold in thresholds
    ]
)
best_threshold = float(thresholds[valid_f1.argmax()])
print("best threshold:", best_threshold, "validation F1:", valid_f1.max())

In [ ]:
test_probability = predict_probability(model, X_test)
test_prediction = test_probability >= best_threshold
precision, recall, f1, _ = precision_recall_fscore_support(
    y_test, test_prediction, average="binary", zero_division=0
)
auc = roc_auc_score(y_test, test_probability)
supervised_metrics = {
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "roc_auc": auc,
}
print(supervised_metrics)
assert all(np.isfinite(value) for value in supervised_metrics.values())

## 4) 정상 센서만 학습하는 autoencoder

레이블이 부족할 때는 정상 패턴을 압축·복원하도록 학습하고 복원 오차가 큰 샘플을
이상으로 간주할 수 있습니다. train에는 시간상 앞부분의 정상 행만 사용합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=DataFrame.groupby -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>frame.groupby(by, ...)</code></summary>

#### `frame.groupby(by, ...)`

- **역할:** key가 같은 행을 그룹으로 묶습니다.
- **입력·반환:** 표 데이터와 열·그룹 설정을 받고 DataFrame, Series 또는 집계 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 학습 전 데이터 분포·결측·중복과 분할 기준을 점검합니다.
- **주의점:** dtype 추론, 결측값, index 보존과 grouping 후 shape 변화를 확인해야 합니다.
- **공식 문서:** [DataFrame.groupby](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
sensor = pd.read_csv(DATA / "sensor_timeseries.csv", parse_dates=["timestamp"])
sensor_features = ["load", "temperature", "vibration", "pressure"]
ordered = sensor.sort_values(["device_id", "timestamp"]).reset_index(drop=True)
train_parts, valid_parts, test_parts = [], [], []
for _, group in ordered.groupby("device_id", sort=False):
    a, b = int(len(group) * 0.60), int(len(group) * 0.80)
    train_parts.append(group.iloc[:a])
    valid_parts.append(group.iloc[a:b])
    test_parts.append(group.iloc[b:])
ae_train = pd.concat(train_parts)
ae_valid = pd.concat(valid_parts)
ae_test = pd.concat(test_parts)
ae_means = ae_train[sensor_features].mean()
ae_stds = ae_train[sensor_features].std().replace(0, 1)
normal_train = ae_train.query("anomaly == 0")
print(
    len(normal_train),
    len(ae_valid),
    len(ae_test),
    "test anomalies:",
    int(ae_test.anomaly.sum()),
)

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.optim.Adam -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.optim.Adam(params, lr=1e-3, ...)</code></summary>

#### `torch.optim.Adam(params, lr=1e-3, ...)`

- **역할:** 1·2차 gradient 모멘트를 추적하는 Adam optimizer를 만듭니다.
- **입력·반환:** 파라미터와 학습률 등을 받고 optimizer 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 노트북의 trainable parameter를 반복 update합니다.
- **주의점:** weight decay의 해석은 AdamW와 다르며 학습률이 결과에 크게 작용합니다.

##### 관련 수식과 코드 연결

$$
\begin{aligned}m_t&=\beta_1m_{t-1}+(1-\beta_1)g_t,\\v_t&=\beta_2v_{t-1}+(1-\beta_2)g_t^2,\\\theta_t&=\theta_{t-1}-\eta\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}\end{aligned}
$$

- **기호:** $g_t$는 gradient, $m_t,v_t$는 1·2차 모멘트, $\eta$는 학습률입니다.
- **수식 → 코드:** `backward()`가 만든 gradient를 읽어 모멘트를 갱신한 뒤 `step()`에서 파라미터에 적용합니다.
- **직관:** 최근 gradient 방향과 크기를 함께 누적해 파라미터별 적응형 보폭을 만듭니다.
- **shape 확인:** 각 trainable parameter와 같은 shape의 모멘트 상태가 optimizer에 저장됩니다.
- **공식 문서:** [torch.optim.Adam](https://docs.pytorch.org/docs/stable/generated/torch.optim.Adam.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
def sensor_matrix(frame):
    return ((frame[sensor_features] - ae_means) / ae_stds).to_numpy(np.float32)


ae_train_x = sensor_matrix(normal_train)
ae_valid_x = sensor_matrix(ae_valid)
ae_test_x = sensor_matrix(ae_test)


class AutoEncoder(nn.Module):
    def __init__(self, input_dim, latent_dim=2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 8), nn.ReLU(), nn.Linear(8, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 8), nn.ReLU(), nn.Linear(8, input_dim)
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))


autoencoder = AutoEncoder(len(sensor_features)).to(device)
ae_optimizer = torch.optim.Adam(autoencoder.parameters(), lr=3e-3)
ae_loader = DataLoader(
    TensorDataset(torch.from_numpy(ae_train_x)), batch_size=128, shuffle=True
)

<!-- api-explanations:start -->
<!-- api-explanations:keys=F.mse_loss -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>F.mse_loss(input, target, reduction=&#x27;mean&#x27;)</code></summary>

#### `F.mse_loss(input, target, reduction='mean')`

- **역할:** 예측과 target의 제곱 오차를 계산합니다.
- **입력·반환:** 예측·target과 설정을 받고 축약된 loss 또는 같은 shape의 loss를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 objective를 수치화해 역전파 시작점으로 사용합니다.
- **주의점:** 입력 형식, target dtype·shape, `reduction` 의미를 반드시 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\mathcal{L}_{MSE}=\frac{1}{N}\sum_{i=1}^{N}(\hat y_i-y_i)^2
$$

- **기호:** $\hat y_i$는 예측, $y_i$는 target입니다.
- **수식 → 코드:** `reduction='mean'`이 모든 원소의 제곱 오차를 평균냅니다.
- **직관:** 큰 오차를 제곱해 더 강하게 벌점하므로 회귀와 reconstruction에 자주 씁니다.
- **shape 확인:** 예측과 target은 같거나 broadcast 가능한 shape이어야 합니다.
- **공식 문서:** [F.mse_loss](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.mse_loss.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
for epoch in range(10):
    autoencoder.train()
    total = 0.0
    for (x,) in ae_loader:
        x = x.to(device)
        ae_optimizer.zero_grad(set_to_none=True)
        loss = nn.functional.mse_loss(autoencoder(x), x)
        loss.backward()
        ae_optimizer.step()
        total += loss.item() * len(x)
    if epoch in {0, 4, 9}:
        print(f"AE epoch={epoch + 1:02d} loss={total / len(ae_loader.dataset):.4f}")


@torch.inference_mode()
def reconstruction_error(features):
    autoencoder.eval()
    x = torch.from_numpy(features).to(device)
    return ((autoencoder(x) - x) ** 2).mean(dim=1).cpu().numpy()


valid_error = reconstruction_error(ae_valid_x)
test_error = reconstruction_error(ae_test_x)

## 5) 정상 validation 오차로 anomaly threshold 설정

threshold 선택에도 test label을 사용하면 평가가 낙관적으로 변합니다. validation 중
정상 샘플 오차의 99% 분위수를 사용하고 test label은 마지막 평가에만 사용합니다.

In [ ]:
valid_normal_error = valid_error[ae_valid.anomaly.to_numpy() == 0]
anomaly_threshold = float(np.quantile(valid_normal_error, 0.99))
anomaly_prediction = test_error >= anomaly_threshold
anomaly_true = ae_test.anomaly.to_numpy()
ae_precision, ae_recall, ae_f1, _ = precision_recall_fscore_support(
    anomaly_true, anomaly_prediction, average="binary", zero_division=0
)
ae_auc = roc_auc_score(anomaly_true, test_error)
unsupervised_metrics = {
    "threshold": anomaly_threshold,
    "precision": ae_precision,
    "recall": ae_recall,
    "f1": ae_f1,
    "roc_auc": ae_auc,
}
print(unsupervised_metrics)

In [ ]:
assert anomaly_threshold > 0
assert all(np.isfinite(v) for v in unsupervised_metrics.values())
print(
    "detected/actual anomalies:", int(anomaly_prediction.sum()), int(anomaly_true.sum())
)

## 지도 분류와 비지도 이상 탐지 비교

| 방식 | 학습 데이터 | score | 장점 | 주의점 |
|---|---|---|---|---|
| MLP | 정상+위험 label | risk probability | 목표 label에 직접 최적화 | label 비용, 분포 변화 |
| Autoencoder | 주로 정상 | reconstruction error | 이상 label 없이 시작 | 잘 복원되는 이상, threshold 민감도 |

두 결과의 수치는 데이터셋과 target 정의가 달라 직접적인 승패 비교가 아닙니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=pd.DataFrame -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>pd.DataFrame(data=None, index=None, columns=None, ...)</code></summary>

#### `pd.DataFrame(data=None, index=None, columns=None, ...)`

- **역할:** 행·열 구조의 DataFrame을 만듭니다.
- **입력·반환:** 표 데이터와 열·그룹 설정을 받고 DataFrame, Series 또는 집계 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 학습 전 데이터 분포·결측·중복과 분할 기준을 점검합니다.
- **주의점:** dtype 추론, 결측값, index 보존과 grouping 후 shape 변화를 확인해야 합니다.
- **공식 문서:** [pd.DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
comparison = pd.DataFrame(
    [
        {"task": "account risk / supervised MLP", **supervised_metrics},
        {
            "task": "sensor anomaly / autoencoder",
            **{k: v for k, v in unsupervised_metrics.items() if k != "threshold"},
        },
    ]
).set_index("task")
print(comparison)

## 운영 threshold 실험

false negative 비용이 크면 threshold를 낮춰 recall을 높이고, 검토 인력이 제한되면
precision 또는 상위 K alert를 최적화할 수 있습니다. 여러 threshold의 alert 수와
precision/recall을 표로 만들어 운영팀과 합의하세요.

In [ ]:
operating_points = []
for threshold in (0.3, 0.5, best_threshold, 0.7):
    prediction = test_probability >= threshold
    p, r, score, _ = precision_recall_fscore_support(
        y_test, prediction, average="binary", zero_division=0
    )
    operating_points.append(
        {
            "threshold": round(float(threshold), 3),
            "alerts": int(prediction.sum()),
            "precision": p,
            "recall": r,
            "f1": score,
        }
    )
print(pd.DataFrame(operating_points).drop_duplicates("threshold"))

## 마무리

불균형 문제에서는 모델 구조보다 split, train-only fit, 손실 가중치, validation threshold,
운영 비용의 정의가 성능을 좌우합니다. autoencoder는 좋은 시작점이지만 시계열 문맥까지
필요하다면 sequence autoencoder나 forecasting residual로 확장해 보세요.